# Dependency Parsing

[← Back to wiki](https://ml-viz-ruby.vercel.app/wiki/dependency-parsing)

Implements a **transition-based arc-standard parser**: the stack/buffer data structure, the SHIFT / LEFT-ARC / RIGHT-ARC actions, and a trace that builds a dependency tree step by step. Then we evaluate a predicted parse with UAS.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

mpl.rcParams.update({
    'figure.facecolor': '#0f1117', 'axes.facecolor': '#1a1d27',
    'text.color': '#e2e8f0', 'axes.labelcolor': '#94a3b8',
    'xtick.color': '#94a3b8', 'ytick.color': '#94a3b8',
    'axes.edgecolor': '#2d3748', 'grid.color': '#2d3748', 'axes.grid': False,
})

## 1. The arc-standard transition system

Configuration = (stack, buffer, arcs). Start: stack=[ROOT], buffer=[all words]. Goal: buffer empty, stack=[ROOT].

In [ ]:
def arc_standard_parse(words, oracle_actions, verbose=True):
    """
    Run the arc-standard parser given a list of oracle actions.
    words: list of tokens (index 0 = ROOT)
    Returns arcs as list of (head_idx, dependent_idx).
    """
    stack = [0]                 # ROOT
    buffer = list(range(1, len(words)))
    arcs = []

    def fmt(idxs): return '[' + ', '.join(words[i] for i in idxs) + ']'

    for action in oracle_actions:
        if verbose:
            print(f"stack={fmt(stack):35s} buffer={fmt(buffer):30s} → {action}")
        if action == 'SHIFT':
            stack.append(buffer.pop(0))
        elif action == 'LEFT-ARC':
            # top is head of second-top; remove second-top
            dependent = stack[-2]
            head = stack[-1]
            arcs.append((head, dependent))
            del stack[-2]
        elif action == 'RIGHT-ARC':
            # second-top is head of top; remove top
            dependent = stack[-1]
            head = stack[-2]
            arcs.append((head, dependent))
            stack.pop()
    return arcs

# Parse: "The cat sat"  (ROOT The cat sat)
words = ['ROOT', 'The', 'cat', 'sat']
# Oracle: build  The<-cat, cat<-sat (subj), sat<-ROOT
oracle = ['SHIFT', 'SHIFT', 'LEFT-ARC', 'SHIFT', 'LEFT-ARC', 'RIGHT-ARC']
arcs = arc_standard_parse(words, oracle)
print("\nFinal arcs (head → dependent):")
for h, d in arcs:
    print(f"  {words[h]} → {words[d]}")

## 2. Visualize the dependency tree

In [ ]:
def plot_dependency_tree(words, arcs, title='Dependency parse'):
    fig, ax = plt.subplots(figsize=(8, 3.5))
    x = np.arange(len(words))
    ax.scatter(x, np.zeros_like(x), s=10, color='#6366f1')
    for i, w in enumerate(words):
        ax.text(i, -0.08, w, ha='center', va='top', color='#e2e8f0', fontsize=11)
    for h, d in arcs:
        # Draw an arc from head to dependent
        mid = (h + d) / 2
        height = 0.15 + 0.15 * abs(h - d)
        xs = np.linspace(h, d, 50)
        ys = height * np.sin(np.pi * (xs - h) / (d - h + 1e-9))
        ax.plot(xs, ys, color='#22d3ee', linewidth=1.5)
        ax.annotate('', xy=(d, 0.02), xytext=(d, ys[len(ys)//2]),
                    arrowprops=dict(arrowstyle='->', color='#22d3ee'))
    ax.set_ylim(-0.3, 1.0)
    ax.set_xlim(-0.5, len(words)-0.5)
    ax.axis('off')
    ax.set_title(title, color='#e2e8f0')
    plt.tight_layout()
    plt.show()

plot_dependency_tree(words, arcs, 'Parse of "The cat sat"')

## 3. UAS evaluation

Unlabeled Attachment Score = fraction of words assigned the correct head.

In [ ]:
def arcs_to_heads(arcs, n_words):
    """Convert arc list to head array: head[d] = h."""
    heads = [-1] * n_words
    for h, d in arcs:
        heads[d] = h
    return heads

def uas(pred_arcs, gold_arcs, n_words):
    pred_heads = arcs_to_heads(pred_arcs, n_words)
    gold_heads = arcs_to_heads(gold_arcs, n_words)
    # exclude ROOT (index 0)
    correct = sum(1 for d in range(1, n_words) if pred_heads[d] == gold_heads[d])
    return correct / (n_words - 1)

gold_arcs = arcs  # our oracle parse is the gold standard here
# A wrong parse: attach 'cat' to ROOT instead of 'sat'
wrong_arcs = [(2, 1), (0, 2), (0, 3)]
print(f"UAS (perfect parse): {uas(arcs, gold_arcs, len(words)):.3f}")
print(f"UAS (wrong parse):   {uas(wrong_arcs, gold_arcs, len(words)):.3f}")

## ✏️ Your turn

**Exercise 1 — Parse a longer sentence.** Find an oracle action sequence that parses `ROOT she ate pizza` into: `she <- ate` (subject), `pizza <- ate` (object), `ate <- ROOT`. Run the parser and verify UAS = 1.0 against the gold arcs `[(2,1),(2,3),(0,2)]`.

In [ ]:
words2 = ['ROOT', 'she', 'ate', 'pizza']
gold2 = [(2,1), (2,3), (0,2)]
# TODO(you): find the oracle action list and run arc_standard_parse(words2, oracle2)
# oracle2 = [...]

In [ ]:
# Assert cell
oracle2_ref = ['SHIFT', 'SHIFT', 'LEFT-ARC', 'SHIFT', 'RIGHT-ARC', 'RIGHT-ARC']
arcs2 = arc_standard_parse(words2, oracle2_ref, verbose=False)
score = uas(arcs2, gold2, len(words2))
print(f"Predicted arcs: {[(words2[h], words2[d]) for h,d in arcs2]}")
print(f"UAS: {score:.3f}")
assert score == 1.0, "Oracle should produce the gold tree"

<details><summary>Solution</summary>

```python
oracle2 = ['SHIFT', 'SHIFT', 'LEFT-ARC',   # she <- ate
           'SHIFT', 'RIGHT-ARC',           # pizza <- ate
           'RIGHT-ARC']                    # ate <- ROOT
arcs2 = arc_standard_parse(words2, oracle2, verbose=False)
assert uas(arcs2, gold2, len(words2)) == 1.0
```

The arc-standard system always reduces the tree bottom-up: dependents must collect all *their* children before being attached to their head (the 'arc-eager' variant relaxes this). A real parser replaces our hand-written oracle with a neural classifier that predicts the next action from the configuration's features.

</details>